# Revenue & growth forecast

Projects subscribers + revenue per channel for the exec review. Run `src/metrics.py` first so the monthly table is fresh.

Last run: refreshing for the June review.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

m = pd.read_csv('../data/processed/monthly_channel_metrics.csv')
daily = pd.read_csv('../data/raw/channel_stats_daily.csv', parse_dates=['date'])
daily['month'] = daily['date'].dt.to_period('M').astype(str)
m['month'].unique()

In [2]:
# company revenue by month (this goes in the deck)
by_month = m.groupby('month')['total_revenue'].sum()
by_month.plot(marker='o', title='Milepost Media — total revenue / month')
plt.show()
by_month

## Subscriber trend + growth baseline

Pull end-of-month subs from the daily feed and get each channel's typical monthly growth.

In [7]:
subs = daily.groupby(['channel_id', 'month'])['subscribers'].last().reset_index()
pivot = subs.pivot(index='channel_id', columns='month', values='subscribers')
pivot

In [8]:
# monthly growth rate per channel, then a single baseline rate per channel
growth = pivot.pct_change(axis=1)
baseline = growth.mean(axis=1)   # avg monthly growth, equal weight across the months we have
baseline.sort_values(ascending=False)

In [12]:
GROWTH_CAP = 0.08      # cap monthly growth so projections don't run away
FORECAST_MONTHS = 12
FLAT_AFTER = 4         # assume growth plateaus after ~4 months, keep revenue flat past that

# base each channel's revenue on its most recent month
base_rev = m[m['month'] == '2026-05'].set_index('channel_id')['ad_revenue']
cur_subs = pivot['2026-06']

rows = []
fc_months = pd.period_range('2026-07', periods=FORECAST_MONTHS, freq='M').astype(str)
for cid in pivot.index:
    g = min(baseline.get(cid, 0.0), GROWTH_CAP)
    s = cur_subs.get(cid, np.nan)
    r = base_rev.get(cid, 0.0)
    for i, mo in enumerate(fc_months):
        if i < FLAT_AFTER:
            s = s * (1 + g)
            rev = r * (1 + g) ** i
        else:
            rev = r * (1 + g) ** FLAT_AFTER
        rows.append([cid, mo, int(s), round(rev, 2)])

forecast = pd.DataFrame(rows, columns=['channel_id', 'forecast_month', 'forecast_subscribers', 'forecast_revenue'])
forecast.head(15)

## Reconcile against actuals

Quick sanity check — first forecast month should land near the latest actual month.

In [14]:
actual_latest = m[m['month'] == m['month'].max()].set_index('channel_id')['total_revenue']
fc_first = forecast[forecast['forecast_month'] == fc_months[0]].set_index('channel_id')['forecast_revenue']
chk = pd.concat([actual_latest, fc_first], axis=1)
chk['delta_pct'] = (chk['forecast_revenue'] / chk['total_revenue'] - 1) * 100
chk

In [15]:
# looks roughly fine, ship it
forecast.to_csv('../outputs/forecast.csv', index=False)
print('company forecast, next 12mo: $%0.0f' % forecast['forecast_revenue'].sum())

# Charts for the exec report
Saved to `../charts/` and referenced from the report doc.

In [ ]:
import matplotlib.ticker as mtick
import os

os.makedirs('../charts', exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})
ACCENT, MUTED = '#2563eb', '#94a3b8'

# 1) company revenue by month -> the plateau
by_month = forecast.assign(m=pd.to_datetime(forecast['forecast_month'])).groupby('m')['forecast_revenue'].sum()
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(by_month.index, by_month.values, marker='o', color=ACCENT, lw=2)
ax.set_title('Total forecast revenue by month', fontweight='bold')
ax.set_ylabel('Revenue / month (USD)'); ax.set_ylim(0, by_month.max() * 1.2)
ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
fig.autofmt_xdate(); fig.tight_layout(); fig.savefig('../charts/revenue_by_month.png'); plt.show()

# 2) revenue by channel (12mo total) -> concentration
ids = forecast.groupby('channel_id')['forecast_revenue'].sum()
names = m.drop_duplicates('channel_id').set_index('channel_id')  # channel_id has no title here; use forecast file titles instead
by_ch = forecast.groupby('channel_id')['forecast_revenue'].sum().sort_values()
total = by_ch.sum()
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = [ACCENT if v >= by_ch.quantile(0.6) else MUTED for v in by_ch.values]
ax.barh([c[-6:] for c in by_ch.index], by_ch.values, color=colors)
ax.set_title('12-month forecast revenue by channel', fontweight='bold')
ax.set_xlabel('12-month forecast revenue (USD)')
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
fig.tight_layout(); fig.savefig('../charts/revenue_by_channel.png'); plt.show()

# 3) subscriber growth by channel -> smaller channels grow fastest
start = forecast[forecast['forecast_month'] == fc_months[0]].set_index('channel_id')['forecast_subscribers']
end = forecast[forecast['forecast_month'] == fc_months[-1]].set_index('channel_id')['forecast_subscribers']
growth = ((end / start - 1) * 100).sort_values()
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh([c[-6:] for c in growth.index], growth.values, color=ACCENT)
port = (end.sum() / start.sum() - 1) * 100
ax.axvline(port, color='#ef4444', ls='--', lw=1.2)
ax.set_title('Forecast subscriber growth by channel', fontweight='bold')
ax.set_xlabel('Subscriber growth, first to last forecast month')
ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('{x:.0f}%'))
fig.tight_layout(); fig.savefig('../charts/subscriber_growth_by_channel.png'); plt.show()
